# Standardise features with broadcasting

A feature matrix has to be standardised before it reaches a model: centre each column, divide by its spread, and clip outliers so one bad sensor reading can't dominate.

Write `standardize(x, clip)`:

- `x`: float64 tensor `(N, D)`, one row per sample. $N \ge 1$.
- `clip`: positive float.

Return a tuple `(mean, std, z, row_norm)`:

1. `mean`: `(D,)`, the column means.
2. `std`: `(D,)`, the **population** standard deviation of each column (divide by $N$, not $N-1$).
3. `z`: `(N, D)`, $z_{ij} = (x_{ij} - \text{mean}_j) / \text{std}_j$, then clamped to `[-clip, clip]`. A column whose `std` is exactly 0 gets `z = 0` everywhere (not `nan`).
4. `row_norm`: `(N,)`, the L2 norm of each row of the clipped `z`.

Use tensor operations only, no Python loops.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch

def standardize(x, clip):
    """x: (N, D) float64. Return (mean (D,), std (D,), z (N, D), row_norm (N,))."""
    mean = x.mean(dim=0)
    std = x.std(dim=0)
    z = (x - mean) / std
    return mean, std, z, z.norm(dim=1)


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 10.0], [3.0, 30.0]], "dtype": "float64"}, 3.0], "e": {"__tup__": [{"__nd__": [2.0, 20.0], "dtype": "float64"}, {"__nd__": [1.0, 10.0], "dtype": "float64"}, {"__nd__": [[-1.0, -1.0], [1.0, 1.0]], "dtype": "float64"}, {"__nd__": [1.4142135623730951, 1.4142135623730951], "dtype": "float64"}]}}, {"a": [{"__nd__": [[0.0, 5.0], [2.0, 5.0], [4.0, 5.0]], "dtype": "float64"}, 1.0], "e": {"__tup__": [{"__nd__": [2.0, 5.0], "dtype": "float64"}, {"__nd__": [1.632993161855452, 0.0], "dtype": "float64"}, {"__nd__": [[-1.0, 0.0], [0.0, 0.0], [1.0, 0.0]], "dtype": "float64"}, {"__nd__": [1.0, 0.0, 1.0], "dtype": "float64"}]}}, {"a": [{"__nd__": [[7.0, -1.0, 2.0]], "dtype": "float64"}, 2.0], "e": {"__tup__": [{"__nd__": [7.0, -1.0, 2.0], "dtype": "float64"}, {"__nd__": [0.0, 0.0, 0.0], "dtype": "float64"}, {"__nd__": [[0.0, 0.0, 0.0]], "dtype": "float64"}, {"__nd__": [0.0], "dtype": "float64"}]}}, {"a": [{"__nd__": [[0.0], [0.0], [0.0], [100.0]], "dtype": "float64"}, 1.5], "e": {"__tup__": [{"__nd__": [25.0], "dtype": "float64"}, {"__nd__": [43.30127018922193], "dtype": "float64"}, {"__nd__": [[-0.5773502691896257], [-0.5773502691896257], [-0.5773502691896257], [1.5]], "dtype": "float64"}, {"__nd__": [0.5773502691896257, 0.5773502691896257, 0.5773502691896257, 1.5], "dtype": "float64"}]}}, {"a": [{"__nd__": [[7.0409191213851825, -20.556650313141816, 5.041809884672578, 3.2966911816162106], [4.547350707889554, 2.844028369102341, 4.798001387085275, 4.304202867067431], [4.134786923725058, 38.22999516644883, 5.022578661322792, 3.942107616975214], [4.718712581848649, -1.6804634610895004, 4.894484944879488, 3.827597068296036], [5.4819453885067855, 2.614463934266333, 5.095775870295976, 4.40059361280026], [5.024259565076664, 20.45820851212812, 5.054510552268765, 3.4843137931579458]], "dtype": "float64"}, 2.0], "e": {"__tup__": [{"__nd__": [5.157995714738649, 6.984930367952384, 4.984526883420812, 3.8759176899855166], "dtype": "float64"}, {"__nd__": [0.9381757590662458, 18.383230859718505, 0.10400532237949785, 0.39880779953364026], "dtype": "float64"}, {"__nd__": [[2.0, -1.4981904373209798, 0.5507699023589369, -1.4523951363204146], [-0.6508855094026971, -0.22525431086891393, -1.7934226063444783, 1.073913743870468], [-1.0906365690283635, 1.6996503518302053, 0.3658637561175565, 0.16596948973189338], [-0.4682311695275657, -0.47137491201449094, -0.86574356466851, -0.1211626797319062], [0.345297424962844, -0.23774201972639394, 1.0696470558423514, 1.3156109871178334], [-0.1425491421725613, 0.7329113281005735, 0.6728854566941692, -0.9819364046678786]], "dtype": "float64"}, {"__nd__": [2.942341534192062, 2.200919540813963, 2.059053569905556, 1.0980104965429867, 1.7466335257079335, 1.4051523808594142], "dtype": "float64"}]}}, {"a": [{"__nd__": [[-0.1828389745977349, 0.5405251317548021, 1.9350880340988528], [-0.2696203273419135, -0.24355867907910456, 1.0023136012756912], [-0.8864599431605871, -0.291720232439864, 0.8825389674564839], [0.5803500161908991, 0.09151670328235219, 0.6701043548284794], [-2.8281623068437627, 1.02130681750008, -0.9596447598081417], [-1.6686198426559695, 0.27644575952099965, 0.7005448853493901], [-0.4447674556827841, -1.0764058401008076, 0.026124833534033623], [-0.05274730824287927, 1.4055981660180925, 0.7474079874793504], [0.19381564626462, 1.1116332052239921, -0.20552304990579248], [50.0, -50.0, 0.0]], "dtype": "float64"}, 2.5], "e": {"__tup__": [{"__nd__": [4.444094950392989, -4.716465896831945, 0.4798954854308347], "dtype": "float64"}, {"__nd__": [15.214302160758606, 15.111201251663358, 0.7541444485785996], "dtype": "float64"}, {"__nd__": [[-0.30411739402183785, 0.3478870369758385, 1.929593927808848], [-0.30982132653397165, 0.29599944724847654, 0.6927295120046332], [-0.35036473163405263, 0.2928123046409053, 0.5339076390266423], [-0.2539547915754973, 0.3181734211623355, 0.25221808601276263], [-0.47798822321234524, 0.37970328227217826, -1.9088388808698393], [-0.40177424691979235, 0.33041130041222583, 0.2925824095561959], [-0.321333331914844, 0.24088489036107963, -0.6017025687214981], [-0.29556677730736286, 0.4051341756947453, 0.35472316020189415], [-0.2793607790366406, 0.38568072815616905, -0.9088690324890596], [2.5, -2.5, -0.636344252530579]], "dtype": "float64"}, {"__nd__": [1.9841485594781556, 0.8145422665208054, 0.7025324604124163, 0.4788959438231238, 2.0040741496307914, 0.5968237925131453, 0.7234131750319549, 0.6142653665536905, 1.0260774761939913, 3.5923438042215143], "dtype": "float64"}]}}, {"a": [{"__nd__": [[0.3968526898941871, -0.41604276802429796, 0.7422782995871782, -0.4492512461038458, 0.12361943746177984], [-0.20068755773909452, 0.2258189838048783, -0.6067215204557526, -0.6394249183138097, 0.49372077129967584], [0.5044468367385548, 0.1339557489058245, 0.8421593545658599, -0.5884499035419624, 0.7018022491833029], [-0.6620253773458276, 0.9287154417885592, 0.24738545621230346, 0.2137675760200295, 0.9411175262652476], [0.5740654275775421, 0.5798349521771087, -0.8918124842580695, -0.26142738781116015, -0.8302104556690639], [-0.6129448370689712, -0.5722660186157056, 0.7172838643318087, -0.7464900405633892, -0.4064844632211011], [-0.014306042151334797, 0.6989208223790737, 0.9304628416747509, 0.4162895413744834, -0.5726255611389037], [0.0899653629160082, 0.4119180859737739, -0.8962349445467497, 0.3597683344481428, -0.26343632972183784], [0.17940057355841654, 0.3390665622833302, 0.3382549922918716, 0.04609955105171282, 0.10947486493017844], [-0.6037005727729341, -0.009625216929195712, -0.7491810547094937, -0.03850599614787109, 0.07248985113962303], [0.5482096326206369, -0.21267041401162823, -0.9607991628967809, 0.05549986642700655, -0.5895350150998242], [0.48251504638873755, -0.22260553916652648, -0.23905367173439518, 0.818909159538594, -0.21405911208958206], [-0.30229684569688664, -0.3039605292834877, -0.03846885939483613, -0.8134157824971724, 0.09349481589142927], [0.8428549240675633, 0.12584412975947346, 0.4878204147010943, 0.8942813263313545, 0.6842307741978282], [0.48801268722496216, 0.6263274783891217, 0.6402780832539134, -0.49248961047593354, -0.03607390562323487], [-0.31456649868296993, -0.4765334127426917, 0.1431031662452611, -0.36422815431815314, 0.23731309154530456], [0.1648330605435142, -0.7912020512360574, -0.1151646968818325, -0.22029025203956065, 0.41330482220076914], [-0.8237343123792285, -0.6620836550800056, 0.025194911910982443, -0.19081254230436895, 0.33069336036343877], [-0.3342866244882994, -0.6055624633100081, 0.8636908465283903, -0.5122070133539303, -0.7058713851335308], [-0.4402116913635734, -0.3205512453029373, -0.5497680146812769, 0.07263290212782803, 0.8738676913353984]], "dtype": "float64"}, 10.0], "e": {"__tup__": [{"__nd__": [-0.00188020590794985, -0.026135055412069896, 0.046535391087211365, -0.12198722950760028, 0.07284165140584488], "dtype": "float64"}, {"__nd__": [0.4839485408752775, 0.494165933428579, 0.6329688366560591, 0.4743325093885762, 0.5133299398844741], "dtype": "float64"}, {"__nd__": [[0.8239158962665367, -0.7890218370720222, 1.0991740322881294, -0.689946419692159, 0.09891841895557972], [-0.41080266813405064, 0.5098571596566008, -1.032052249197742, -1.0908754482655987, 0.8198998094452676], [1.0462414903261266, 0.3239616361394367, 1.256971777128694, -0.9834086106297912, 1.2252560174436873], [-1.3640813344409055, 1.9322467062344193, 0.3173143028433633, 0.7078469193697565, 1.6914576910413797], [1.1900968488174941, 1.226248040582017, -1.4824550925801072, -0.2939713292755349, -1.759204045799749], [-1.2626644767971393, -1.1051570459633215, 1.0596864085570563, -1.3165928935817306, -0.9337583440677925], [-0.025675945258376787, 1.4672316093515878, 1.3964786248518672, 1.1348089372493841, -1.2574119730674822], [0.1897837498545705, 0.886449493486088, -1.4894419456960424, 1.0156494746201044, -0.6550913067789553], [0.37458689128083517, 0.7390262925685509, 0.46087513999235646, 0.3543648753402549, 0.0713638747285575], [-1.2435627262694535, 0.03340950350083233, -1.2571178859301084, 0.17599728398826833, -0.0006853297243893914], [1.1366701044984764, -0.3774751474780039, -1.5914441527732825, 0.3741828620673526, -1.290352685554905], [1.0009230556219924, -0.3975799836935788, -0.4511897684100192, 1.983621974928575, -0.5589012859058923], [-0.6207615364344279, -0.5622108993710541, -0.1342945269329853, -1.4576874646033369, 0.04023370327908875], [1.745506099569411, 0.3075468681483052, 0.6971670610912986, 2.1425235161489242, 1.1910256450842858], [1.0122830254780466, 1.3203308639151488, 0.9380283163755949, -0.7811026519053439, -0.2121745656479562], [-0.6461147546999323, -0.9114314177946407, 0.1525632378178556, -0.5106985501010802, 0.320401027410313], [0.3444855235020311, -1.5481985787969363, -0.25546295268389024, -0.20724496125866404, 0.6632443275596709], [-1.6982262307989593, -1.2869130724079918, -0.033714897069766575, -0.14509929518743261, 0.5023118445334086], [-0.6868631486710419, -1.1725361233984817, 1.290988447011338, -0.8226713879454117, -1.5169834759971852], [-0.9057398637112323, -0.5957840676069568, -0.94207387668361, 0.41030316873346384, 1.560450653063069]], "dtype": "float64"}, {"__nd__": [1.7307187328992675, 1.831963411651643, 2.29082850733328, 3.009494969152981, 2.8807821655424575, 2.5581146288300807, 2.6405409660499606, 2.121541034560225, 1.0146666262281516, 1.7773253337147794, 2.4025452695431277, 2.3686738485140033, 1.6869891806983972, 3.10424993119306, 2.0743738899537383, 1.2786386802214689, 1.7503410946134121, 2.1942254386947315, 2.5478030133795504, 2.160134596727088], "dtype": "float64"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(standardize, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")